# Lab 3｜false negative 诊断与 positive mask

Lab 1 写了模型和训练目标，Lab 2 写了训练循环和评估。这一章用它们回答 Lab 1 留下的那个问题：

> in-batch 的其他列，对这个用户来说真的都是负样本吗？

先量，再改，最后做一次单变量对照。

符号沿用前两个 lab：📖 概念，🧩 要你写的代码，✅ 断言，🚀 可以改着玩的地方，💡 折叠的参考答案。

In [ ]:
# 不用改：把前两个 lab 导出的东西接上
import math
import sys
import time
from pathlib import Path

import torch
from torch.nn import functional as F


def find_labs_dir() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        for candidate in (base, base / 'labs'):
            if (candidate / 'labkit.py').exists():
                return candidate
    raise FileNotFoundError('找不到 labs 目录，请从仓库根目录或 labs 目录启动')


LABS_DIR = find_labs_dir()
if str(LABS_DIR) not in sys.path:
    sys.path.insert(0, str(LABS_DIR))

import labkit  # noqa: E402

try:
    from myrec.eval import group_seen_items, ndcg_at_k, recall_at_k, recommend_top_k
    from myrec.model import TwoTower, in_batch_cross_entropy_loss, train_one_step
    from myrec.train import train_one_epoch
except ModuleNotFoundError as error:
    raise SystemExit('先跑完 Lab 1 和 Lab 2：它们会把代码导出成 labs/myrec/') from error

PASSED = []


def check(name: str, actual, expected) -> None:
    '''严格相等：列表、元组、整数、布尔。'''
    ok = actual == expected
    print(('PASS' if ok else 'FAIL'), name, '| 期望:', expected, '| 实际:', actual)
    if not ok:
        raise AssertionError('还没对，改那一格，或看下面的参考答案')
    PASSED.append(name)


def check_close(name: str, actual: float, expected: float, tolerance: float = 1e-5) -> None:
    '''浮点比较，允许误差。'''
    ok = abs(actual - expected) <= tolerance
    print(('PASS' if ok else 'FAIL'), name, '| 期望约:', expected, '| 实际:', actual)
    if not ok:
        raise AssertionError('还没对，改那一格，或看下面的参考答案')
    PASSED.append(name)


print('labs dir:', LABS_DIR)
print('train_one_epoch 来自:', train_one_epoch.__module__)
print('recommend_top_k 来自:', recommend_top_k.__module__)

In [ ]:
# 不用改：数据、以及评估要用到的几组索引
labkit.set_seed(42)
data = labkit.prepare_movielens()
print(data.describe())

# 正式评估还要过滤此前播放过但未达正反馈阈值的物品，labkit 已按目标时间构造。
seen_items = data.validation_seen
validation_users = [row.user_idx for row in data.validation]
test_users = [row.user_idx for row in data.test]
print('validation / test 用户数:', len(validation_users), '/', len(test_users))

## 📖 理解一下：把那个假设变成一个可数的东西

Lab 1 里说过，in-batch 训练默认「同一批里其他用户的物品，对我来说都是负样本」。

但 `item_ids` 是从全体正反馈里抽出来的。用户 A 的正样本，完全可能同时是用户 B 在训练集里喜欢过的东西。这时候 loss 一边说「这是正反馈」，一边要求把它压低——目标本身自相矛盾。

要确认这不是少数噪声，得先有一张能回答「(用户, 物品) 在训练集里是不是正反馈」的表。

> 完整推导与文献：[04｜真凶在训练目标里](../docs/01-two-tower/04-false-negatives.md)

### 🧩 你来写：`build_train_positive_matrix()`

返回 `[num_users, num_items]` 的布尔张量，第 `(u, i)` 位为 `True` 表示训练集里有这条正反馈。

In [ ]:
def build_train_positive_matrix(interactions, num_users, num_items):
    '''返回 [num_users, num_items] 的布尔真值表。'''
    # TODO 3-0：思路是「先全 False，再把出现过的地方设成 True」
    #   提示：torch.zeros(..., dtype=torch.bool)
    #   取出所有 user_idx 和 item_idx 之后，可以直接用花式索引赋值
    raise NotImplementedError('TODO 3-0')

In [ ]:
# ✅ 跑一下：三条手造交互，其中两条是同一个 (用户, 物品)
sample = [
    labkit.Interaction(user_idx=0, item_idx=2, timestamp=1),
    labkit.Interaction(user_idx=0, item_idx=2, timestamp=9),
    labkit.Interaction(user_idx=1, item_idx=0, timestamp=2),
]
sample_matrix = build_train_positive_matrix(sample, 3, 4)

check('3-0 形状是 [3, 4]', list(sample_matrix.shape), [3, 4])
check('3-0 dtype 是 bool', sample_matrix.dtype, torch.bool)
check('3-0 重复的 (0, 2) 只算一个，共 2 个 True', int(sample_matrix.sum()), 2)
check('3-0 (0, 2) 是 True', bool(sample_matrix[0, 2]), True)
check('3-0 (2, 3) 是 False', bool(sample_matrix[2, 3]), False)

<details>
<summary>💡 参考答案（<code>build_train_positive_matrix</code>）</summary>

```python
def build_train_positive_matrix(interactions, num_users, num_items):
    '''返回 [num_users, num_items] 的布尔真值表。'''
    positive_matrix = torch.zeros((num_users, num_items), dtype=torch.bool)
    user_indices = torch.tensor([row.user_idx for row in interactions])
    item_indices = torch.tensor([row.item_idx for row in interactions])
    positive_matrix[user_indices, item_indices] = True
    return positive_matrix
```

</details>

`positive_matrix[u, i] = True` 这种按坐标批量赋值，PyTorch 和 NumPy 是同一套写法，比 Python 里双重循环快好几个数量级。

这张表是 `6038 × 3533` 的 bool，不到 22 MB，在这个数据规模上很直观。换成上亿用户和物品就不能这么存了——那时候要么按 batch 现查，要么换成稀疏结构。

## 📖 理解一下：怎么算「冲突」

一个 batch 里有 `user_ids` 和 `item_ids`，各 256 个。用真值表切出这批用户和这批物品之间的所有已知关系：

```text
known = positive_matrix[user_ids][:, item_ids]     # [B, B]
```

它和 `logits` 完全对齐。对角线是本轮明确给出的正样本，本来就该是正类；要找的是**非对角线上的 True**——那才是被当成负样本的正反馈。

```text
conflict = known & ~diagonal
```

顺便留意一下候选位置的规模：每个 batch 的非对角位置是 `B × (B - 1)`，累计起来是千万级。所以这个比例只要不是零头，就值得处理。

### 🧩 你来写：`diagnose_false_negatives()`

扫一遍 dataloader，返回四个数：批次数、有冲突的批次数、冲突位置数、非对角位置总数。

In [ ]:
def diagnose_false_negatives(dataloader, positive_matrix, max_batches=None):
    '''返回 (批次数, 有冲突的批次数, 冲突位置数, 非对角位置总数)。'''
    # TODO 3-1：按上面那个 known & ~diagonal 的思路累加
    #   每个 batch 的非对角位置数是 B * (B - 1)
    #   不涉及模型，纯统计，用不着梯度
    raise NotImplementedError('TODO 3-1')

In [ ]:
# ✅ 跑一下：扫完整训练集。几千万个位置，CPU 上要跑一小会儿
positive_matrix = build_train_positive_matrix(data.train, data.num_users, data.num_items)
print('真值表:', list(positive_matrix.shape), '| True 个数:', f'{int(positive_matrix.sum()):,}')

started = time.time()
batches, with_conflicts, conflict_cells, off_diagonal_cells = diagnose_false_negatives(
    labkit.create_train_loader(data.train, batch_size=256, seed=42), positive_matrix
)
ratio = conflict_cells / off_diagonal_cells
print()
print(f'批次数          {batches:,}')
print(f'有冲突的批次    {with_conflicts:,}')
print(f'冲突位置        {conflict_cells:,}')
print(f'非对角位置      {off_diagonal_cells:,}')
print(f'冲突比例        {ratio:.4%}   ({time.time() - started:.0f}s)')

check('3-1 每个 batch 都有冲突', with_conflicts, batches)
check('3-1 冲突比例远高于噪声（> 10%）', ratio > 0.10, True)

<details>
<summary>💡 参考答案（<code>diagnose_false_negatives</code>）</summary>

```python
def diagnose_false_negatives(dataloader, positive_matrix, max_batches=None):
    '''返回 (批次数, 有冲突的批次数, 冲突位置数, 非对角位置总数)。'''
    batches = 0
    batches_with_conflicts = 0
    conflict_cells = 0
    off_diagonal_cells = 0
    for batch_number, batch in enumerate(dataloader):
        if max_batches is not None and batch_number >= max_batches:
            break
        user_ids = batch['user_idx']
        item_ids = batch['item_idx']
        known = positive_matrix[user_ids][:, item_ids]
        diagonal = torch.eye(len(user_ids), dtype=torch.bool)
        conflicts = known & ~diagonal
        batches += 1
        count = int(conflicts.sum())
        conflict_cells += count
        batches_with_conflicts += count > 0
        off_diagonal_cells += len(user_ids) * (len(user_ids) - 1)
    return batches, batches_with_conflicts, conflict_cells, off_diagonal_cells
```

</details>

`batches_with_conflicts += count > 0` 用到了 Python 里 `bool` 就是 `int`：`True` 加 1，`False` 加 0。写成 `if count > 0: batches_with_conflicts += 1` 也完全可以，看你觉得哪个清楚。

## 📖 理解一下：mask 要怎么加

想要的改动很小：把那些「非对角、但已知是正样本」的位置，从 CrossEntropy 的候选里拿掉。做法是把它们的 logit 设成 `-inf`，softmax 之后这些位置的概率就是 0，不再参与竞争。

关键是**不能连对角线一起屏蔽**。对角线上是每一行的正确类别，把它也设成 `-inf`，那一行就没有任何合法候选了，softmax 直接给出 NaN。

另外注意这里仍然是「每行一个标签」的 CrossEntropy——没有把这些位置变成第二个正类，只是让它们退出负类竞争。

> 完整推导与文献：[04｜真凶在训练目标里](../docs/01-two-tower/04-false-negatives.md)

### 🧩 你来写：`masked_in_batch_loss()`

输入 `logits` 和 `[B, B]` 的已知正样本矩阵。四行左右。

In [ ]:
def masked_in_batch_loss(logits, known_positive_mask):
    '''屏蔽「非对角且已知正样本」的位置，对角线保留。返回标量 loss。'''
    # TODO 3-2：三步
    #   1) 造 [B, B] 的对角线
    #   2) false_negative_mask = known_positive_mask & ~diagonal
    #   3) masked_fill 成 -inf 之后，交给 Lab 1 的 in_batch_cross_entropy_loss
    #   提示：torch.eye(size, dtype=torch.bool, device=logits.device)
    raise NotImplementedError('TODO 3-2')

In [ ]:
# ✅ 跑一下：三种情况——没有冲突、有冲突、以及「对角线也被屏蔽」的反例
torch.manual_seed(0)
small_logits = torch.randn(4, 4)
plain_loss = in_batch_cross_entropy_loss(small_logits).item()

no_conflict = torch.zeros(4, 4, dtype=torch.bool)
check_close(
    '3-2 没有冲突时，masked loss 与普通 loss 相同',
    masked_in_batch_loss(small_logits, no_conflict).item(),
    plain_loss,
)

with_conflict = torch.zeros(4, 4, dtype=torch.bool)
with_conflict[1, 3] = True
masked_loss = masked_in_batch_loss(small_logits, with_conflict).item()
check('3-2 屏蔽一个冲突位置会改变 loss', bool(masked_loss != plain_loss), True)

# 把整张矩阵都当冲突：如果实现里忘了保住对角线，这里会变成 NaN
all_conflict = torch.ones(4, 4, dtype=torch.bool)
check(
    '3-2 对角线保住了（loss 仍是有限值）',
    bool(torch.isfinite(torch.tensor(masked_in_batch_loss(small_logits, all_conflict).item()))),
    True,
)

print()
print('反例：如果把对角线也一起屏蔽掉会怎样')
broken = F.cross_entropy(
    small_logits.masked_fill(torch.ones(4, 4, dtype=torch.bool), -torch.inf),
    torch.arange(4),
)
print(' ->', broken.item())

<details>
<summary>💡 参考答案（<code>masked_in_batch_loss</code>）</summary>

```python
def masked_in_batch_loss(logits, known_positive_mask):
    '''屏蔽「非对角且已知正样本」的位置，对角线保留。返回标量 loss。'''
    batch_size = logits.shape[0]
    diagonal = torch.eye(batch_size, dtype=torch.bool, device=logits.device)
    false_negative_mask = known_positive_mask & ~diagonal
    masked_logits = logits.masked_fill(false_negative_mask, -torch.inf)
    return in_batch_cross_entropy_loss(masked_logits)
```

</details>

上面那个反例打印出来是 `nan`。这是这类改动最常见的翻车方式，而且不会报错——loss 变成 NaN 之后，梯度一路传回去把参数也变成 NaN，最后只看到指标全是 0。所以加完 mask 之后第一件事就是确认 loss 还是有限值。

### 🧩 你来写：`train_one_epoch_masked()`

和 Lab 2 的 `train_one_epoch` 只差 loss 那一行。这正是「单变量对照」的意思：除了训练目标，其他完全不动。

In [ ]:
def train_one_epoch_masked(model, dataloader, optimizer, positive_matrix, max_batches=None):
    '''训练一个 epoch，loss 用 masked_in_batch_loss。返回平均 loss。'''
    # TODO 3-3：和 train_one_epoch 一样的循环，开头同样要 model.train()
    #   区别只有两处：先从 positive_matrix 切出 batch 的 [B, B] 子矩阵，再用 masked_in_batch_loss
    raise NotImplementedError('TODO 3-3')

In [ ]:
# ✅ 跑一下：把 mask 全设成 False，两个训练循环应该完全等价
# 只用 512 条交互、向量压到 8 维，跑得很快
labkit.set_seed(7)
model_a = TwoTower(data.num_users, data.num_items, embedding_dim=8, temperature=0.1)
model_b = TwoTower(data.num_users, data.num_items, embedding_dim=8, temperature=0.1)
model_b.load_state_dict(model_a.state_dict())

subset = data.train[:512]
loader_a = labkit.create_train_loader(subset, batch_size=64, seed=1)
loader_b = labkit.create_train_loader(subset, batch_size=64, seed=1)
empty_matrix = torch.zeros((data.num_users, data.num_items), dtype=torch.bool)

loss_a = train_one_epoch(model_a, loader_a, torch.optim.Adam(model_a.parameters(), lr=1e-2))
loss_b = train_one_epoch_masked(
    model_b, loader_b, torch.optim.Adam(model_b.parameters(), lr=1e-2), empty_matrix
)

check_close('3-3 零 mask 时两个训练循环的 loss 相同', loss_b, loss_a)
check(
    '3-3 跑完之后两份参数也完全相同',
    bool(torch.equal(model_a.user_embedding.weight.detach(), model_b.user_embedding.weight.detach())),
    True,
)

<details>
<summary>💡 参考答案（<code>train_one_epoch_masked</code>）</summary>

```python
def train_one_epoch_masked(model, dataloader, optimizer, positive_matrix, max_batches=None):
    '''训练一个 epoch，loss 用 masked_in_batch_loss。返回平均 loss。'''
    model.train()
    losses = []
    for batch_number, batch in enumerate(dataloader):
        if max_batches is not None and batch_number >= max_batches:
            break
        user_ids = batch['user_idx']
        item_ids = batch['item_idx']
        logits = model(user_ids, item_ids)
        known_positive_mask = positive_matrix[user_ids][:, item_ids]
        loss = masked_in_batch_loss(logits, known_positive_mask)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return sum(losses) / len(losses)
```

</details>

和 Lab 2 那版逐行对比，只多了「切子矩阵」和「换 loss」两处。上面那格用全 `False` 的 mask 把两版对齐跑了一遍，loss 和最终参数都完全相同——这也是加完新逻辑之后值得做的一种自检：**让它退化成一个已知正确的实现，看结果对不对得上**。

In [ ]:
# ✅ 跑一下：单变量对照。两组只差训练目标，其他全部相同
# 每组各自建一个 seed=42 的 loader，保证两组看到的 batch 顺序一模一样
def run_config(use_mask, epochs=5):
    labkit.set_seed(42)
    model = TwoTower(data.num_users, data.num_items, embedding_dim=64, temperature=0.1)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
    loader = labkit.create_train_loader(data.train, batch_size=256, seed=42)

    for epoch in range(1, epochs + 1):
        if use_mask:
            loss = train_one_epoch_masked(model, loader, optimizer, positive_matrix)
        else:
            loss = train_one_epoch(model, loader, optimizer)
        print(f'  epoch {epoch}/{epochs}  loss = {loss:.4f}')

    recommendations = recommend_top_k(
        model, validation_users, data.num_items, seen_items, k=20
    )
    return (
        model,
        recall_at_k(recommendations, data.validation, 20),
        ndcg_at_k(recommendations, data.validation, 20),
    )


print('A) 不加 mask')
plain_model, plain_recall, plain_ndcg = run_config(use_mask=False)
print(f'   -> Recall@20 = {plain_recall:.6f}   NDCG@20 = {plain_ndcg:.6f}')

print()
print('B) 加 positive mask')
masked_model, masked_recall, masked_ndcg = run_config(use_mask=True)
print(f'   -> Recall@20 = {masked_recall:.6f}   NDCG@20 = {masked_ndcg:.6f}')

print()
print(
    f'Recall@20   {plain_recall:.6f}  ->  {masked_recall:.6f}'
    f'   ({(masked_recall / plain_recall - 1):+.1%})'
)
print(
    f'NDCG@20     {plain_ndcg:.6f}  ->  {masked_ndcg:.6f}'
    f'   ({(masked_ndcg / plain_ndcg - 1):+.1%})'
)

check('3-4 加 mask 之后 Recall@20 更高', masked_recall > plain_recall, True)
check('3-4 两组都明显高于随机水平', min(plain_recall, masked_recall) > 5 * 20 / 3533, True)

# A 组和 Lab 2 是同一个配置。数值对上，才说明两章用的确实是同一套 seed、loader 和超参
check_close(
    '3-4 不加 mask 的基线与 Lab 2 记录的一致',
    plain_recall,
    labkit.recorded('no_mask_recall20'),
    tolerance=1e-12,
)

## 📖 收束：这个差距说明什么

看上面两组的结果，训练目标只差一行：

| | 训练 loss | Recall@20 | NDCG@20 |
|---|---|---|---|
| A 不加 mask | 6.15 → 4.76 | 0.079339 | 0.030552 |
| B 加 mask | 5.68 → 3.74 | **0.105785** (+33.3%) | **0.041346** (+35.3%) |

（正文里的 mask 对照是另一套配置：MLP tower + lr=0.001，0.048 → 0.086。和这里的 0.079339 → 0.105785 都是真实结果，但属于两套实验，数字不能直接横向比。）

两组的 loss 曲线都记在上面了，但这两个 loss 不能横向比较。mask 之后每行参加 softmax 的候选变少了，loss 天然会低一些，这是结构决定的，跟「目标对不对」无关——B 组的 loss 更低，说明不了任何问题。

能说明问题的是右边两列：两组用同一个 seed、同一份初始参数、同一批 batch 顺序，只有 mask 这一个变量不同，而 validation 上的两个指标都明显提高。

这说明 Lab 2 里那个偏低、而且怎么训都不太动的 Recall，有很大一部分不是模型容量不够，而是**训练目标本身写错了**。反过来说：在改模型结构、加层数、换优化器之前，先确认目标是对的。这类错误不会让 loss 报错，它照样平稳下降，只是优化的东西和你想的不是一回事。

mask 也不是唯一解法。多正样本 loss、按用户分组采样都能处理同一类问题，只是改动更大。这里的实现胜在只有几行，而且行为一眼能看懂。

In [ ]:
# ✅ 跑一下：配置已经由 validation 锁定（加 mask），现在只跑一次 test
# test 只用来看最终结果，拿它回头挑配置就失去意义了
test_recommendations = recommend_top_k(
    masked_model, test_users, data.num_items, data.test_seen, k=20
)
test_recall = recall_at_k(test_recommendations, data.test, 20)
test_ndcg = ndcg_at_k(test_recommendations, data.test, 20)

print(f'test        Recall@20 = {test_recall:.6f}   NDCG@20 = {test_ndcg:.6f}')
print(f'validation  Recall@20 = {masked_recall:.6f}   NDCG@20 = {masked_ndcg:.6f}')

check('3-5 test 每个用户都有推荐', len(test_recommendations), len(test_users))
check('3-5 test 指标落在 [0, 1]', 0.0 <= test_recall <= 1.0, True)

In [ ]:
# 💾 导出这一章写的东西
labkit.harvest(
    'myrec/mask.py',
    build_train_positive_matrix,
    diagnose_false_negatives,
    masked_in_batch_loss,
    train_one_epoch_masked,
    imports=['import torch', 'from myrec.model import in_batch_cross_entropy_loss'],
)

import importlib

import myrec.mask as my_mask

importlib.reload(my_mask)

check('3-6 交接成功：myrec.mask 能用', hasattr(my_mask, 'masked_in_batch_loss'), True)

## 收尾

### 这个 lab 写了多少

| 自己写的 | 行数 | 脚手架给的 |
|---|---|---|
| `build_train_positive_matrix` | 6 | `labkit.py` 的数据管道 |
| `diagnose_false_negatives` | 18 | 前两个 lab 导出的 `myrec.model` / `train` / `eval` |
| `masked_in_batch_loss` | 5 | `check()` / `check_close()` |
| `train_one_epoch_masked` | 18 | |

加起来 47 行左右。

### 三个 lab 走完之后

`labs/myrec/` 里是这套东西：

```text
myrec/model.py    TwoTower、in_batch_cross_entropy_loss、train_one_step
train.py    train_one_epoch
eval.py     group_seen_items、recommend_top_k、recall_at_k、ndcg_at_k
mask.py     build_train_positive_matrix、diagnose_false_negatives、
            masked_in_batch_loss、train_one_epoch_masked
```

训练、评估、修目标三件事都是自己写的，彼此之间是 import 关系，不是复制粘贴。

### 这几条应该能直接答上来

1. 为什么同一批里其他用户的物品，不一定是这个用户的负样本？
2. 加 mask 时为什么必须保住对角线？坏了会看到什么现象？
3. 「零 mask 时两个训练循环等价」这个自检，验证的是什么？
4. 为什么候选配置只能在 validation 上挑，test 只跑一次？

### 再往下

这里只用了 MovieLens 和 ID 特征，模型也只有查表。仓库的 `src/recsys/` 是同一套算法的工程版本，多了统一配置、checkpoint 元数据和测试；另外还有序列模型和 Popularity / MF 两个基线可以对照。

真正想继续练的话，比加模型更值当的方向是：把 mask 换成多正样本 loss 做同样的对照，或者把 `temperature`、`embedding_dim`、学习率按同一套纪律各消融一次。